# Layer 2: Normalization

Takes Layer 1 output (per-frame joint coordinates) and produces:
- **Normalized coordinates**: hip center = origin, shoulder width = 1.0
- **Joint angles**: 6 three-point angles + 2 line angles (degrees)

**Input**: `sample_videos/chon_ji_master_poses.json`  
**Goal**: Verify normalization is consistent across frames and inspect
joint angles at the approximate completion of Chon-Ji Movement 1.

In [ ]:
import sys
import pathlib
import json

project_root = pathlib.Path().resolve().parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"project_root: {project_root}")

In [ ]:
from itf_analysis.pose_extraction.extractor import Landmark
from itf_analysis.normalization.normalizer import (
    normalize_pose,
    extract_joint_angles,
)

JSON_PATH = str(
    project_root / "itf_analysis" / "sample_videos" / "chon_ji_master_poses.json"
)

# Adjust this frame index after visually inspecting the overlay video.
# Rough estimate: Movement 1 of Chon-Ji completes around frame 55-70.
MOVEMENT_1_END_FRAME = 60

In [ ]:
# Load Layer 1 JSON and reconstruct Landmark objects
with open(JSON_PATH, encoding="utf-8") as f:
    raw_data = json.load(f)

def load_landmarks(frame_dict: dict):
    return {
        int(k): Landmark(**v)
        for k, v in frame_dict["landmarks"].items()
    }

print(f"Loaded {len(raw_data)} frames")

## Normalize all frames and extract angles

In [ ]:
import math

all_angles = []   # list of (frame_index, Dict[str, float])
skipped = []

for frame_dict in raw_data:
    fi = frame_dict["frame_index"]
    lms = load_landmarks(frame_dict)
    normalized = normalize_pose(lms)

    if normalized is None:
        skipped.append(fi)
        continue

    angles = extract_joint_angles(normalized)
    all_angles.append((fi, angles))

print(f"Normalized : {len(all_angles)} frames")
print(f"Skipped    : {len(skipped)} frames (anchor joint missing)")

## Normalization sanity check

After normalization, the hip center must be at (0,0) and the
shoulder width must be 1.0 in every frame.

In [ ]:
import numpy as np

hip_center_errors = []
shoulder_width_errors = []

for frame_dict in raw_data:
    lms = load_landmarks(frame_dict)
    n = normalize_pose(lms)
    if n is None:
        continue

    # Hip center should be at origin
    hx = (n[23].x + n[24].x) / 2
    hy = (n[23].y + n[24].y) / 2
    hip_center_errors.append(math.sqrt(hx**2 + hy**2))

    # Shoulder width should be 1.0
    sw = math.sqrt(
        (n[11].x - n[12].x)**2 +
        (n[11].y - n[12].y)**2 +
        (n[11].z - n[12].z)**2
    )
    shoulder_width_errors.append(abs(sw - 1.0))

print(f"Hip center offset   — max: {max(hip_center_errors):.2e}  mean: {np.mean(hip_center_errors):.2e}")
print(f"Shoulder width error— max: {max(shoulder_width_errors):.2e}  mean: {np.mean(shoulder_width_errors):.2e}")

## Joint angles over time

In [ ]:
import matplotlib.pyplot as plt

frame_indices = [fi for fi, _ in all_angles]
angle_names = ["right_knee", "left_knee", "right_hip", "left_hip",
               "right_elbow", "left_elbow"]

fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex=True)
axes = axes.flatten()

for ax, name in zip(axes, angle_names):
    values = [a.get(name) for _, a in all_angles]
    valid_fi   = [fi for fi, v in zip(frame_indices, values) if v is not None]
    valid_vals = [v  for v in values if v is not None]
    ax.plot(valid_fi, valid_vals, linewidth=0.8)
    ax.axvline(MOVEMENT_1_END_FRAME, color='red', linestyle='--',
               label=f'approx. movement 1 end (frame {MOVEMENT_1_END_FRAME})')
    ax.set_title(name)
    ax.set_ylabel('degrees')
    ax.legend(fontsize=7)

fig.supxlabel('frame index')
plt.suptitle('Joint angles over time — Chon-Ji master', y=1.01)
plt.tight_layout()
plt.show()

## Movement 1 completion: joint angles

Adjust `MOVEMENT_1_END_FRAME` above after comparing with the overlay video,
then re-run this cell.

In [ ]:
# Find the closest normalized frame to the target
target_entry = min(all_angles, key=lambda x: abs(x[0] - MOVEMENT_1_END_FRAME))
target_frame, target_angles = target_entry

print(f"=== Chon-Ji Movement 1 completion — frame {target_frame} ===")
print()

angle_labels = {
    "right_knee":         "Right knee",
    "left_knee":          "Left knee",
    "right_elbow":        "Right elbow",
    "left_elbow":         "Left elbow",
    "right_hip":          "Right hip",
    "left_hip":           "Left hip",
    "shoulder_line_angle":"Shoulder tilt",
    "hip_line_angle":     "Hip tilt",
}

for key, label in angle_labels.items():
    val = target_angles.get(key)
    if val is not None:
        print(f"  {label:22s}: {val:7.2f} deg")
    else:
        print(f"  {label:22s}: — (joint missing)")

print()
print("Reference ranges for walking stance (ap-kubi):")
print("  Front knee angle: 100–120 deg  (right_knee here)")

## Validation Checklist

- [ ] Hip center offset < 1e-9 in every normalized frame
- [ ] Shoulder width error < 1e-9 in every normalized frame  
- [ ] Skipped frames < 5% of total
- [ ] Joint angle curves look smooth (no wild spikes)
- [ ] Movement 1 front knee angle in 100–120° range

All items passing → proceed to Layer 3 (segmentation).